# Q-switching and mode-locking {#sec-pulsed}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/pulsed.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/pulsed.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter4/laserlib.py', 'laserlib.py')
else:
    sys.path.insert(0, '..')                 # laserlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import laserlib as lz
lz.use_style()

*Slides 84–87 (Q-switching: principle, time sequence, commercial Q-switched Nd:YAG lasers, rotating prism, saturable absorber
and electro-optic switches) and 88 (mode-locking).*

## Key relations

**Q-switching** (slides 84–87). The quality factor $Q$ of the cavity is kept low (a mirror is blocked or misaligned) while the
pump builds up an inversion $N_i$ far above the normal threshold $N_{th}$. The medium stores energy because its upper level is
long-lived (ms for Nd:YAG and ruby, @sec-einstein). Then $Q$ is switched high in nanoseconds. The photon density grows from
spontaneous noise at the rate $(N/N_{th}-1)/\tau_c$, where $\tau_c$ is the photon lifetime of the high-$Q$ cavity, and a giant
pulse empties the inversion. Without pumping during the pulse, the four-level rate equations in normalised form ($n=N/N_{th}$,
time in units of $\tau_c$) are

$$
\frac{dn}{dt}=-n\phi,\qquad \frac{d\phi}{dt}=(n-1)\phi .
$$

Their solution gives the fraction $\eta$ of the stored inversion that is extracted, from $r\eta=-\ln(1-\eta)$ with
$r=N_i/N_{th}$, and the peak photon density $\phi_\text{max}=r-1-\ln r$ (in units of $N_{th}$), reached when $n=1$.

**Mode-locking** (slide 88). A laser oscillating on $N$ longitudinal modes spaced by $\delta\nu=c/2L$ emits their sum. If the
phases are random, the output is noisy around its average. If the phases are **locked** (equal), the modes add in phase once per
round trip $T=2L/c$ and produce a train of pulses of width

$$
\Delta t\approx\frac{T}{N}\approx\frac{1}{\Delta\nu},\qquad P_\text{peak}\approx N\,P_\text{average}.
$$

The pulse width is set by the **gain bandwidth** $\Delta\nu$: the broader the gain, the shorter the pulse.

## The giant pulse

```{ojs}
//| echo: false
import {rk4} from "./laser_sim.js"
viewof qs = Inputs.form({
  r: Inputs.range([1.05, 10], {value: 3, step: 0.05, label: "initial inversion r = N_i/N_th"}),
  tc: Inputs.range([1, 50], {value: 10, step: 1, label: "photon lifetime τ_c (ns)"}),
  phi0: Inputs.range([-12, -3], {value: -8, step: 0.5, label: "log₁₀ initial photon density (spontaneous seed)"})
})
```

```{ojs}
//| echo: false
qsOut = {
  const tEnd = 30*Math.max(1, 4/(qs.r - 1)) + 20;
  const sol = rk4((t, [n, p]) => [-n*p, (n - 1)*p], [qs.r, 10 ** qs.phi0], 0, tEnd, 6000, 10);
  const data = sol.flatMap(({t, y}) => [{t: t*qs.tc, v: y[0], who: "inversion n = N/N_th"}, {t: t*qs.tc, v: y[1], who: "photon density φ (units of N_th)"}]);
  const ph = sol.map(d => d.y[1]), pk = d3.max(ph), ipk = ph.indexOf(pk);
  const above = sol.filter(d => d.y[1] >= pk/2), fw = (above[above.length - 1].t - above[0].t)*qs.tc;
  // solve r η = −ln(1 − η) by bisection
  let a = 1e-9, b = 1 - 1e-12; for (let i = 0; i < 100; i++) { const c = (a + b)/2; if (qs.r*c + Math.log(1 - c) > 0) a = c; else b = c; }
  return {data, pk, tpk: sol[ipk].t*qs.tc, fw, eta: (a + b)/2, nf: sol[sol.length - 1].y[0]};
}
Plot.plot({
  width: Math.min(width, 1050), height: 300, x: {label: "time after the Q-switch (ns)"}, y: {label: "normalised value", grid: true},
  color: {legend: true, range: ["#eb6834", "#2a78d6"]},
  marks: [Plot.line(qsOut.data, {x: "t", y: "v", stroke: "who", strokeWidth: 2}), Plot.ruleY([1], {stroke: "#52514e", strokeDasharray: "3,3"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">build-up delay (peak) ≈ <b>${qsOut.tpk.toFixed(0)} ns</b> · pulse FWHM ≈ <b>${qsOut.fw.toFixed(1)} ns</b> ·
peak φ = ${qsOut.pk.toFixed(3)} (formula r − 1 − ln r = ${(qs.r - 1 - Math.log(qs.r)).toFixed(3)}) ·
final inversion n<sub>f</sub> = ${qsOut.nf.toFixed(3)} · extracted fraction η = <b>${(100*qsOut.eta).toFixed(1)} %</b></div>`
```

Things to see:

* The inversion falls **below** threshold at the end ($n_f<1$): the photons present at the peak keep stimulating emission
  while they leak out.
* A larger $r$ gives a shorter, stronger and earlier pulse. Near $r=1$ the pulse is long and weak: Q-switching only pays off when
  the stored inversion is many times threshold.
* The build-up delay depends only logarithmically on the spontaneous seed.

In [ ]:
#| label: fig-qswitch
#| fig-cap: "Q-switched pulses for four initial inversions (τ_c = 10 ns). Inset numbers: energy extraction efficiency η from rη = −ln(1 − η)."
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for r, c in zip((1.5, 2, 3, 5), lz.PALETTE):
    t, n, phi = lz.qswitch_pulse(r, 10e-9, t_end=600e-9)
    eta = lz.qswitch_energy_fraction(r)
    ax.plot(t*1e9, phi, color=c, label=f"r = {r}: η = {eta:.0%}, φ_max = {r - 1 - np.log(r):.2f}")
ax.set(xlabel="time (ns)", ylabel="photon density φ / N_th"); ax.legend(fontsize=8); plt.show()

### Commercial Q-switched lasers (slide 86)

In [ ]:
#| label: tbl-qsw
#| tbl-cap: "Slide 86 recomputed: peak power ≈ energy/width and average power = energy × repetition rate."
q = pd.DataFrame({"laser": ["NL220 (EKSPLA), EO switch", "P-1064-150-HE (Alphalas), passive"],
                  "energy (mJ)": [10, 1.5], "width (ns)": [7, 1.1], "rep. rate (Hz)": [1000, 100]})
q["peak power ≈ E/Δt (kW)"] = (q["energy (mJ)"]*1e-3/(q["width (ns)"]*1e-9)/1e3).round(0)
q["average power (W)"] = q["energy (mJ)"]*1e-3*q["rep. rate (Hz)"]
q

Both lasers reach about 1.4 MW peak, as the slide says. The average power is only 10 W and 0.15 W: a Q-switch does not create
energy. It compresses the energy stored over a millisecond into nanoseconds, a factor of $10^5$ to $10^6$ in power.

## Mode-locking (slide 88)

```{ojs}
//| echo: false
viewof ml = Inputs.form({
  N: Inputs.range([1, 101], {value: 15, step: 1, label: "number of oscillating modes N"}),
  L: Inputs.range([0.1, 2], {value: 1.5, step: 0.05, label: "cavity length L (m)"}),
  locked: Inputs.radio(["locked (equal phases)", "random phases"], {value: "locked (equal phases)", label: "phases"}),
  seed: Inputs.range([1, 50], {value: 7, step: 1, label: "random seed"}),
  shape: Inputs.radio(["equal amplitudes", "Gaussian gain envelope"], {value: "equal amplitudes", label: "mode amplitudes"})
})
```

```{ojs}
//| echo: false
mlOut = {
  const c = 299792458, T = 2*ml.L/c, dnu = 1/T;
  let s = ml.seed*9301 + 49297; const rnd = () => (s = (s*9301 + 49297) % 233280)/233280;
  const ph = d3.range(ml.N).map(() => ml.locked.startsWith("locked") ? 0 : 2*Math.PI*rnd());
  const amp = d3.range(ml.N).map(k => ml.shape.startsWith("equal") ? 1 : Math.exp(-((k - (ml.N - 1)/2)**2)/(2*(ml.N/4 + 0.5)**2)));
  const P = amp.reduce((a, b) => a + b*b, 0), data = [];
  for (let i = 0; i <= 1500; i++) {
    const t = 3*T*i/1500; let re = 0, im = 0;
    for (let k = 0; k < ml.N; k++) { const a = 2*Math.PI*(k - (ml.N - 1)/2)*dnu*t + ph[k]; re += amp[k]*Math.cos(a); im += amp[k]*Math.sin(a); }
    data.push({t: t*1e9, I: (re*re + im*im)/P});
  }
  const peak = d3.max(data, d => d.I);
  return {data, T, dnu, peak};
}
Plot.plot({
  width: Math.min(width, 1050), height: 280, x: {label: "time (ns)"}, y: {label: "intensity / average intensity", grid: true},
  marks: [Plot.line(mlOut.data, {x: "t", y: "I", stroke: "#4a3aa7", strokeWidth: 1.5}), Plot.ruleY([1], {stroke: "#eda100", strokeDasharray: "4,3"})]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">round trip T = 2L/c = <b>${(mlOut.T*1e9).toFixed(2)} ns</b> (repetition rate ${(mlOut.dnu/1e6).toFixed(1)} MHz) ·
total bandwidth N·c/2L = ${(ml.N*mlOut.dnu/1e9).toFixed(2)} GHz · expected pulse width ≈ T/N = <b>${(mlOut.T/ml.N*1e9).toFixed(3)} ns</b> ·
peak/average = <b>${mlOut.peak.toFixed(1)}</b> (dashed: average = CW laser)</div>`
```

With random phases the intensity fluctuates wildly, but the average is the same: mode-locking redistributes the power in time.
With locked phases the peak-to-average ratio equals $N$, and the pulse narrows as $1/N$.

In [ ]:
#| label: tbl-modelock
#| tbl-cap: "Shortest mode-locked pulses allowed by the gain bandwidth (transform limit Δt ≈ 0.44/Δν for Gaussian pulses)."
media = [("He–Ne (Doppler)", 1.5e9), ("Nd:YAG", 120e9), ("Nd:glass", 7e12), ("Er:fiber (C band)", 4e12),
         ("semiconductor (InGaAsP)", 5e12), ("Ti:sapphire (650–1100 nm)", 100e12)]
pd.DataFrame([{"gain medium": m, "gain bandwidth": f"{d/1e9:,.0f} GHz" if d < 1e12 else f"{d/1e12:.0f} THz",
               "shortest pulse": (f"{0.44/d*1e12:.0f} ps" if 0.44/d > 1e-12 else f"{0.44/d*1e15:.0f} fs"),
               "modes in a 1.5 m cavity": f"{d/(lz.c0/3):,.0f}"} for m, d in media])

::: {.callout-tip collapse="true"}
## Going further
- **How are the phases locked?** Active mode-locking modulates the loss (or the gain) at exactly $c/2L$ (the EO switch of slide
  88c): each mode acquires sidebands that coincide with its neighbours and injection-locks them. Passive mode-locking uses a
  saturable absorber, or the Kerr lens of the gain crystal (Ti:sapphire), which favours high peak intensity. Write the field as a
  sum of modes with a sinusoidal loss modulation and iterate it round trip by round trip.
- **Frequency combs.** The mode-locked spectrum is a comb of lines $\nu_m=f_\text{ceo}+mf_\text{rep}$. Measuring and stabilising
  $f_\text{ceo}$ with self-referencing earned the 2005 Nobel prize (Hall, Hänsch). Combs now calibrate astronomical spectrographs
  and link optical clocks.
- **Cavity dumping and regenerative amplifiers.** Chirped-pulse amplification (Strickland and Mourou, Nobel 2018) stretches the
  mode-locked pulse, amplifies it and recompresses it to reach petawatt peak powers.
- **Passive Q-switching.** A saturable absorber (Cr⁴⁺:YAG) switches $Q$ by itself (slide 87b). Add an absorber population to the
  rate equations and show that the laser emits a regular pulse train whose repetition rate grows with the pump.
:::

::: {.callout-note collapse="true"}
## Complements
- Kasap, sec. 4.9; Siegman, *Lasers*, ch. 26 (Q-switching) and 27–28 (mode-locking); Svelto, ch. 8.
- J. J. Degnan, "Theory of the optimally coupled Q-switched laser", *IEEE J. Quantum Electron.* 25, 214 (1989).
- U. Keller, "Recent developments in compact ultrafast lasers", *Nature* 424, 831 (2003).
:::

## Try it yourself

1. A Nd:YAG rod stores $10^{18}$ cm⁻³ × 1 cm³ excess ions above threshold. What is the stored energy? What peak power would it
   give in a 10 ns pulse?
2. For $r=2$, what fraction of the initial inversion remains after the pulse? Where does the "missing" energy below threshold go?
3. A Ti:sapphire laser has a 1.8 m cavity and emits 20 fs pulses at 800 nm with 1 W average power. Find the repetition rate,
   the pulse energy, the peak power and the number of locked modes.
4. In the mode-locking panel, choose the Gaussian envelope. Why does the pulse lose its side lobes?